# Retards et annulations de vols aux États-Unis en 2015

Ce notebook vérifie la qualité des données, puis cherche ce qui explique les retards et les annulations.

In [5]:
import pandas as pd

flights = pd.read_csv("../data/raw/flights.csv", low_memory=False)
len(flights)

5819079

## 1. Nettoyage

### Pourquoi certains vols n'ont-ils pas de retard au départ ?

Hypothèse : ces vols ont été annulés, donc ils ne sont jamais partis. Si c'est vrai, ces cases vides ne sont pas des erreurs, et il ne faut pas les remplacer par 0.

In [6]:
pd.crosstab(flights["DEPARTURE_DELAY"].isna(), flights["CANCELLED"])

CANCELLED,0,1
DEPARTURE_DELAY,,
False,5729195,3731
True,0,86153


**Constat :** tous les retards vides correspondent à des vols annulés. On ne les remplace donc pas par 0.

### Faut-il garder les vols annulés qui ont un retard au départ ?

**Choix : on les garde.** Un long retard au départ pourrait précéder une annulation : les retirer ferait disparaître ce signal. On compare donc leur retard à celui des autres vols.

In [7]:
flights.groupby("CANCELLED")["DEPARTURE_DELAY"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
CANCELLED,,,,,,,,
0,5729195.0,9.3,37.0,-82.0,-5.0,-2.0,7.0,1988.0
1,3731.0,57.5,95.5,-25.0,-2.0,21.0,83.0,1274.0


In [8]:
avec_annules = flights["DEPARTURE_DELAY"].mean()
sans_annules = flights.loc[flights["CANCELLED"] == 0, "DEPARTURE_DELAY"].mean()
round(avec_annules, 2), round(sans_annules, 2)

(np.float64(9.37), np.float64(9.34))

### Quand les causes de retard sont-elles renseignées ?

Hypothèse : les 5 colonnes de causes ne sont remplies que pour les vols arrivés avec au moins 15 minutes de retard.

In [ ]:
causes = ["AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"]

cause_renseignee = flights[causes].notna().all(axis=1)
retard_15_min = flights["ARRIVAL_DELAY"] >= 15

pd.crosstab(retard_15_min, cause_renseignee)

### La somme des causes correspond-elle au retard à l'arrivée ?

In [ ]:
vols_avec_cause = flights[cause_renseignee]
somme_causes = vols_avec_cause[causes].sum(axis=1)

(somme_causes == vols_avec_cause["ARRIVAL_DELAY"]).mean()

**Constat :** les causes sont renseignées pour tous les vols arrivés avec au moins 15 minutes de retard, et seulement pour eux. Leur somme est toujours égale au retard à l'arrivée.

**Conséquence :** on peut calculer la part de chaque cause, mais uniquement pour ces vols. Une case vide veut dire « cause non renseignée », pas « 0 minute ».

### Les codes d'aéroport des vols existent-ils dans la liste des aéroports ?

Hypothèse : les codes d'octobre sont écrits autrement, donc ce mois-là ne se relie pas à `airports.csv`.

In [ ]:
airports = pd.read_csv("../data/raw/airports.csv")

origine_connue = flights["ORIGIN_AIRPORT"].isin(airports["IATA_CODE"])
destination_connue = flights["DESTINATION_AIRPORT"].isin(airports["IATA_CODE"])

pd.crosstab(flights["MONTH"], [origine_connue, destination_connue])

In [ ]:
flights.loc[~origine_connue, "ORIGIN_AIRPORT"].value_counts().head()

In [ ]:
airports[airports[["LATITUDE", "LONGITUDE"]].isna().any(axis=1)]

**Constat :** tous les vols d'octobre ont des codes à 5 chiffres, alors que les 11 autres mois utilisent des codes à 3 lettres. Sans correction, octobre disparaît des analyses par aéroport.

### Traduire les codes d'octobre

**Choix :** on récupère ces vols plutôt que de les exclure, car ils représentent 8,4 % des données. Le bureau américain des statistiques de transport (BTS) publie deux tables qui donnent le nom de chaque aéroport : l'une par numéro, l'autre par code à lettres. Le nom sert de lien entre les deux.

In [9]:
# Ces fichiers ne sont pas en UTF-8. keep_default_na=False évite que le code "NA" soit lu comme une case vide.
table_numeros = pd.read_csv("../data/raw/L_AIRPORT_ID.csv", dtype=str, encoding="latin-1", keep_default_na=False)
table_lettres = pd.read_csv("../data/raw/L_AIRPORT.csv", dtype=str, encoding="latin-1", keep_default_na=False)

table_numeros[table_numeros["Code"] == "10397"]

,Code,Description
373,10397,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern..."


In [10]:
est_octobre = flights["MONTH"] == 10
colonnes_aeroport = ["ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]

codes_octobre = pd.unique(flights.loc[est_octobre, colonnes_aeroport].values.ravel())
codes_autres_mois = pd.unique(flights.loc[~est_octobre, colonnes_aeroport].values.ravel())

correspondance = table_numeros[table_numeros["Code"].isin(codes_octobre)].merge(
    table_lettres[table_lettres["Code"].isin(codes_autres_mois)],
    on="Description",
    suffixes=("_numero", "_lettres"),
)

numeros_sans_traduction = set(codes_octobre) - set(correspondance["Code_numero"])
len(codes_octobre), len(correspondance), numeros_sans_traduction

(307, 306, {'14027'})

### Le code 14027 n'a pas de traduction : lequel est-ce ?

Une route garde la même distance toute l'année. On compare donc les routes de 14027 à celles de PBI (Palm Beach).

In [ ]:
traduction = correspondance.set_index("Code_numero")["Code_lettres"]

routes_14027 = flights[flights["ORIGIN_AIRPORT"] == "14027"]
distances_14027 = routes_14027.groupby(routes_14027["DESTINATION_AIRPORT"].map(traduction))["DISTANCE"].median()

routes_pbi = flights[flights["ORIGIN_AIRPORT"] == "PBI"]
distances_pbi = routes_pbi.groupby("DESTINATION_AIRPORT")["DISTANCE"].median()

pd.DataFrame({"14027": distances_14027, "PBI": distances_pbi.reindex(distances_14027.index)})

**Constat :** les distances sont identiques sur toutes les routes. On ajoute donc 14027 = PBI à la main, puis on traduit les codes.

In [ ]:
traduction["14027"] = "PBI"

for colonne in colonnes_aeroport:
    flights[colonne] = flights[colonne].map(traduction).fillna(flights[colonne])

flights["ORIGIN_AIRPORT"].isin(airports["IATA_CODE"]).mean(), flights["DESTINATION_AIRPORT"].isin(airports["IATA_CODE"]).mean()

### La traduction est-elle juste ?

On compare la distance de chaque vol d'octobre à la distance de la même route les autres mois. Une mauvaise traduction créerait de grands écarts.

In [ ]:
distance_habituelle = (
    flights[~est_octobre]
    .groupby(colonnes_aeroport)["DISTANCE"]
    .median()
    .rename("DISTANCE_HABITUELLE")
)

vols_octobre = flights[est_octobre].join(distance_habituelle, on=colonnes_aeroport)
ecart = (vols_octobre["DISTANCE"] - vols_octobre["DISTANCE_HABITUELLE"]).abs()

ecart.value_counts(dropna=False).sort_index()

### Combien de vols sont touchés par les aéroports sans coordonnées ?

In [ ]:
sans_coordonnees = airports.loc[airports["LATITUDE"].isna(), "IATA_CODE"]

(flights["ORIGIN_AIRPORT"].isin(sans_coordonnees) | flights["DESTINATION_AIRPORT"].isin(sans_coordonnees)).sum()

**Constat :** 3 aéroports (ECP, PBG, UST) n'ont pas de coordonnées, ce qui touche 10 020 vols, soit 0,17 %.

**Choix :** on garde ces vols dans les chiffres. Ces aéroports n'apparaîtront pas sur la carte, et on le précisera sous celle-ci.

### Les horaires sont-ils utilisables ?

Les horaires sont écrits HHMM : 1430 veut dire 14 h 30, et 5 veut dire 00 h 05. On vérifie d'abord que tous les horaires sont valides.

In [ ]:
colonnes_horaires = ["SCHEDULED_DEPARTURE", "DEPARTURE_TIME", "SCHEDULED_ARRIVAL", "ARRIVAL_TIME"]

pd.DataFrame({
    "minimum": flights[colonnes_horaires].min(),
    "maximum": flights[colonnes_horaires].max(),
    "minutes_invalides": (flights[colonnes_horaires] % 100 >= 60).sum(),
    "egal_a_2400": (flights[colonnes_horaires] == 2400).sum(),
})

**Constat :** aucun horaire n'a de minutes invalides. La valeur 2400 existe pour minuit dans les heures réelles et d'arrivée, mais jamais dans le départ prévu : l'heure extraite de cette colonne reste donc entre 0 et 23.

### Les horaires sont-ils cohérents avec le retard ?

Heure prévue + retard doit donner l'heure réelle de départ. On compte en minutes depuis minuit, et le modulo 1440 gère le passage de minuit.

In [ ]:
def en_minutes(horaire):
    return horaire // 100 * 60 + horaire % 100


depart_calcule = en_minutes(flights["SCHEDULED_DEPARTURE"]) + flights["DEPARTURE_DELAY"]
depart_reel = en_minutes(flights["DEPARTURE_TIME"])
a_un_depart = flights["DEPARTURE_TIME"].notna()

(((depart_calcule - depart_reel) % 1440 == 0)[a_un_depart]).mean()

**Constat :** un seul vol sur 5 732 926 ne respecte pas le calcul. C'est un vol annulé du 24 décembre, prévu à 8 h 04 et parti à 11 h 03, avec un retard indiqué de 1 minute. **Choix :** on le garde, car il ne change aucun résultat.

### Créer l'heure de départ prévue

On garde uniquement l'heure, de 0 à 23, pour comparer les moments de la journée. On utilise l'heure prévue, car elle existe pour tous les vols, même annulés.

In [ ]:
flights["HEURE_DEPART_PREVUE"] = flights["SCHEDULED_DEPARTURE"] // 100

flights.groupby("HEURE_DEPART_PREVUE")["DEPARTURE_DELAY"].agg(["count", "mean", "median"]).round(1)

**Constat :** le retard moyen passe d'environ 2 minutes à 5-6 h à 15 minutes vers 18-20 h. Entre 0 h et 4 h, il y a trop peu de vols pour conclure avec la même confiance.

**Hypothèse à vérifier :** les retards s'accumulent au fil de la journée, car un avion en retard repart en retard.

### Les valeurs extrêmes sont-elles possibles ?

On regarde d'abord le plus petit et le plus grand chiffre de chaque colonne. Une valeur impossible serait par exemple une distance nulle ou un temps de vol négatif.

In [ ]:
colonnes_durees = ["DEPARTURE_DELAY", "ARRIVAL_DELAY", "AIR_TIME", "ELAPSED_TIME", "DISTANCE"]

flights[colonnes_durees].describe().round(1)

**Constat :** aucune valeur impossible dans ces colonnes : la distance et le temps de vol sont toujours positifs. Un retard négatif veut dire que le vol est en avance.

### Les plus gros retards sont-ils de vraies données ?

Si le plus gros retard était une erreur de saisie, l'avion ne serait pas arrivé en retard d'autant. On compare donc le retard au départ et à l'arrivée.

In [ ]:
colonnes_vol = ["MONTH", "DAY", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "SCHEDULED_DEPARTURE", "DEPARTURE_TIME", "DEPARTURE_DELAY", "ARRIVAL_DELAY"]

flights.nlargest(5, "DEPARTURE_DELAY")[colonnes_vol]

**Constat :** le plus gros retard est de 1 988 minutes, soit plus de 33 heures. Il est crédible : le vol prévu à 7 h 00 est parti à 16 h 08 le lendemain et il est arrivé avec 1 971 minutes de retard.

### Combien de vols ont un très gros retard, et pèsent-ils sur la moyenne ?

On compare la moyenne avec et sans les vols partis avec plus de 3 heures de retard.

In [ ]:
retard = flights["DEPARTURE_DELAY"]
plus_de_3h = retard > 180

plus_de_3h.sum(), plus_de_3h.mean(), retard.mean(), retard[~plus_de_3h].mean(), retard.quantile([0.5, 0.9, 0.99])

**Constat :** 48 107 vols, soit 0,8 %, partent avec plus de 3 heures de retard. Sans eux, la moyenne passe de 9,4 à 7,1 minutes.

**Choix :** on garde ces vols, car ce sont de vrais retards. Comme ils tirent la moyenne vers le haut, on ne s'appuiera pas sur elle seule : on regardera aussi la médiane et la part de vols en retard.

### Y a-t-il des vols en double ?

Un doublon compterait un même vol deux fois. On cherche d'abord les lignes entièrement identiques, puis les lignes qui décrivent le même vol avec des chiffres différents : même jour, même compagnie, même numéro de vol et même heure prévue au même aéroport.

In [ ]:
colonnes_identite = ["MONTH", "DAY", "AIRLINE", "FLIGHT_NUMBER", "ORIGIN_AIRPORT", "SCHEDULED_DEPARTURE"]

flights.duplicated().sum(), flights.duplicated(subset=colonnes_identite).sum()

In [ ]:
flights[flights.duplicated(subset=colonnes_identite, keep=False)].sort_values(colonnes_identite)[colonnes_identite + ["DESTINATION_AIRPORT", "TAIL_NUMBER", "DEPARTURE_DELAY", "CANCELLED"]].head(10)

**Constat :** aucune ligne n'est entièrement identique. Une seule paire partage la même identité : le vol AA 803 du 29 août, au départ de STT à 14 h 35. Ses deux lignes ont des destinations différentes (CLT et SJU) et sont toutes les deux annulées.

**Choix :** on ne supprime aucune ligne, car rien ne prouve que ce soit un doublon.

## 2. Exploration

### Que devient un vol sur 100 ?

Un vol est dit « en retard » s'il arrive avec au moins 15 minutes de retard. C'est aussi le seuil à partir duquel les causes de retard sont renseignées.

In [ ]:
flights["EN_RETARD"] = flights["ARRIVAL_DELAY"] >= 15

annule = flights["CANCELLED"] == 1
devie = flights["DIVERTED"] == 1
a_l_heure = flights["ARRIVAL_DELAY"] < 15

pd.Series({
    "à l'heure": a_l_heure.mean(),
    "en retard": flights["EN_RETARD"].mean(),
    "annulé": annule.mean(),
    "dévié": devie.mean(),
}).mul(100).round(2)

**Constat :** sur 100 vols, environ 80 arrivent à l'heure, 18 arrivent en retard, 1,5 est annulé et 0,3 est dévié.

### Certains mois sont-ils pires que d'autres ?

La part de vols en retard est calculée sur les vols arrivés, et la part d'annulations sur tous les vols.

In [ ]:
arrive = flights["ARRIVAL_DELAY"].notna()

par_mois = flights.groupby("MONTH").agg(
    vols=("MONTH", "size"),
    pct_annules=("CANCELLED", "mean"),
)
par_mois["pct_annules"] = (par_mois["pct_annules"] * 100).round(2)
par_mois["pct_en_retard"] = (flights[arrive].groupby("MONTH")["EN_RETARD"].mean() * 100).round(1)

par_mois

**Constat :** le risque de retard varie presque du simple au double selon le mois : 23,5 % en juin contre 12,4 % en octobre. Février combine beaucoup de retards (23,3 %) et le plus d'annulations (4,78 %). Septembre et octobre sont les mois les plus calmes.

**Hypothèse à vérifier :** la météo d'hiver expliquerait février, et les orages ou le trafic d'été expliqueraient juin.

### D'où viennent les minutes de retard ?

Chaque vol arrivé avec au moins 15 minutes de retard répartit ses minutes entre 5 causes. On additionne ces minutes sur toute l'année.

In [ ]:
noms_causes = {
    "LATE_AIRCRAFT_DELAY": "Avion arrivé en retard",
    "AIRLINE_DELAY": "Compagnie",
    "AIR_SYSTEM_DELAY": "Système aérien",
    "WEATHER_DELAY": "Météo extrême",
    "SECURITY_DELAY": "Sûreté",
}
colonnes_causes = list(noms_causes)

(flights[colonnes_causes].sum() / flights[colonnes_causes].sum().sum() * 100).rename(noms_causes).round(1)

**Constat :** la première cause est un avion déjà en retard sur son vol précédent (39,8 % des minutes), devant la compagnie (32,2 %) et le système aérien (22,9 %). La météo extrême ne pèse que 4,9 %.

**Limite :** « avion arrivé en retard » ne dit pas ce qui a causé le premier retard de la chaîne.

### Quelle cause augmente selon le mois et selon l'heure ?

On compte les minutes de retard de chaque cause, divisées par le nombre de vols arrivés. Cela permet de comparer des mois et des heures qui n'ont pas le même nombre de vols.

In [ ]:
def minutes_par_vol(colonne):
    groupes = flights[arrive].groupby(colonne)
    return groupes[colonnes_causes].sum().div(groupes.size(), axis=0).rename(columns=noms_causes).round(1)


minutes_par_vol("MONTH")

**Constat :** en juin et en février, toutes les causes augmentent en même temps, et aucune n'explique à elle seule le pic. Le mois de plus fort trafic est juillet, pas juin : le nombre de vols ne suffit donc pas à expliquer juin.

**Conclusion :** ces données ne confirment pas l'hypothèse des orages ou du trafic pour juin. La question reste ouverte.

In [ ]:
minutes_par_vol("HEURE_DEPART_PREVUE")

**Constat :** entre 6 h et 20 h, les minutes dues à un avion déjà en retard passent de 0,3 à 7,8 par vol, alors que les autres causes bougent bien moins. Cela confirme que les retards s'accumulent au fil de la journée. Entre 0 h et 4 h, il y a trop peu de vols pour conclure.

### Pourquoi les vols sont-ils annulés ?

Selon la description du jeu de données, A = compagnie, B = météo, C = système aérien et D = sûreté. On regarde la part de vols annulés par raison, pour chaque mois.

In [ ]:
noms_annulations = {"A": "Compagnie", "B": "Météo", "C": "Système aérien", "D": "Sûreté"}

(pd.crosstab(flights["MONTH"], flights["CANCELLATION_REASON"]).div(flights.groupby("MONTH").size(), axis=0) * 100).rename(columns=noms_annulations).round(2)

In [ ]:
flights["CANCELLATION_REASON"].map(noms_annulations).value_counts(normalize=True).mul(100).round(1)

**Constat :** la météo est la première raison d'annulation sur l'année (54,3 %), et 3,60 points sur les 4,78 % de vols annulés en février. L'hypothèse de la météo d'hiver est donc confirmée pour les annulations de février.

### Certaines compagnies font-elles nettement mieux que d'autres ?

On compare la part de vols en retard, la part d'annulations et les minutes de retard attribuées à la compagnie elle-même.

In [ ]:
airlines = pd.read_csv("../data/raw/airlines.csv")
noms_compagnies = airlines.set_index("IATA_CODE")["AIRLINE"]

par_compagnie = flights.groupby("AIRLINE").agg(
    vols=("AIRLINE", "size"),
    pct_annules=("CANCELLED", "mean"),
)
par_compagnie["pct_annules"] = (par_compagnie["pct_annules"] * 100).round(2)
par_compagnie["pct_en_retard"] = (flights[arrive].groupby("AIRLINE")["EN_RETARD"].mean() * 100).round(1)
par_compagnie["minutes_compagnie"] = minutes_par_vol("AIRLINE")["Compagnie"]

par_compagnie.rename(index=noms_compagnies).sort_values("pct_en_retard")

**Constat :** la part de vols en retard va de 11,3 % (Hawaiian) à 29,7 % (Spirit). Delta reste à 13,6 % avec 875 881 vols : être une grande compagnie n'empêche donc pas d'être ponctuel. American Eagle annule le plus (5,10 %).

Les minutes attribuées à la compagnie elle-même varient bien moins : Spirit (4,1) est au niveau de United (4,4), alors que Spirit a bien plus de vols en retard.

**Limite :** les compagnies ne volent pas aux mêmes endroits. Cet écart ne mesure donc pas uniquement leur qualité.

### Juin : quelques très mauvais jours, ou tout le mois ?

Pour chaque jour de l'année, on calcule la part de vols en retard. Puis on résume les jours de chaque mois : un jour médian, le pire jour, et le nombre de jours avec au moins 30 % de vols en retard.

In [ ]:
retard_par_jour = flights[arrive].groupby(["MONTH", "DAY"])["EN_RETARD"].mean() * 100

retard_par_jour.groupby("MONTH").agg(
    jour_median="median",
    pire_jour="max",
    jours_30_pct=lambda jours: (jours >= 30).sum(),
).round(1)

**Constat :** juin a le jour médian le plus élevé de l'année (23,7 %), mais pas le pire jour : janvier monte à 49,3 %. Juin est donc mauvais sur tout le mois, et pas à cause de quelques jours catastrophiques.

### Juin : le problème vient-il de certains aéroports ?

On compare juin au reste de l'année, aéroport de départ par aéroport de départ, et on regarde les 20 plus gros aéroports.

In [ ]:
gros_aeroports = flights["ORIGIN_AIRPORT"].value_counts().head(20).index
vols_gros_aeroports = flights[arrive & flights["ORIGIN_AIRPORT"].isin(gros_aeroports)]

juin_par_aeroport = pd.DataFrame({
    "juin": vols_gros_aeroports[vols_gros_aeroports["MONTH"] == 6].groupby("ORIGIN_AIRPORT")["EN_RETARD"].mean() * 100,
    "autres_mois": vols_gros_aeroports[vols_gros_aeroports["MONTH"] != 6].groupby("ORIGIN_AIRPORT")["EN_RETARD"].mean() * 100,
})
juin_par_aeroport["ecart"] = juin_par_aeroport["juin"] - juin_par_aeroport["autres_mois"]

juin_par_aeroport.sort_values("ecart", ascending=False).round(1)

**Constat :** juin ne touche pas tous les aéroports de la même façon. ORD, IAH et EWR gagnent environ 10 points ou plus, alors que SEA et SLC bougent à peine et que JFK baisse.

**Conclusion :** c'est compatible avec un phénomène local, comme les orages d'été, sans le prouver. Il faudrait des données météo extérieures à ce fichier pour trancher.

### Nos chiffres et la météo de juin concordent-ils avec des sources extérieures ?

On recalcule le taux de ponctualité comme les chiffres officiels, en comptant les vols annulés et déviés comme non ponctuels. Puis on regarde les pires jours de juin, pour tout le pays et pour Houston.

In [ ]:
ponctualite = flights.groupby("MONTH")["ARRIVAL_DELAY"].apply(lambda retards: (retards < 15).mean() * 100).round(1)
vols_iah = flights[arrive & (flights["MONTH"] == 6) & (flights["ORIGIN_AIRPORT"] == "IAH")]

ponctualite.loc[[5, 6]], retard_par_jour.loc[6].nlargest(5).round(1), (vols_iah.groupby("DAY")["EN_RETARD"].mean() * 100).nlargest(3).round(1)

**Constat :** on retrouve les chiffres publiés, 80,5 % en mai et 74,8 % en juin ([résumé du rapport officiel](https://www.travelpulse.com/news/airlines-airports/airlines-on-time-performance-up-but-so-are-complaints)). Les pires jours nationaux de juin sont du 15 au 18, et Houston atteint son pire jour le 17, pendant la [tempête tropicale Bill (15-22 juin)](https://www.wpc.ncep.noaa.gov/tropical/rain/bill2015.html). L'Illinois a aussi connu [son mois de juin le plus pluvieux jamais enregistré](https://www.weather.gov/ilx/June_2015_hvyrain).

**Conclusion :** la météo est une explication probable d'une partie de juin, mais ce n'est pas une preuve. Le même rapport indique une ponctualité encore plus basse en juin 2014 (71,8 %) : juin 2015 n'est donc pas un cas isolé.

### Quels aéroports posent le plus de problèmes sur l'année ?

Pour chaque aéroport, on regarde la part de vols qui en partent avec au moins 15 minutes de retard, la part de vols qui y arrivent avec au moins 15 minutes de retard, et la part de vols annulés au départ. On affiche les 30 plus gros aéroports : les petits aéroports ont trop peu de vols pour une comparaison fiable.

In [ ]:
par_aeroport = flights.groupby("ORIGIN_AIRPORT").agg(
    vols=("ORIGIN_AIRPORT", "size"),
    pct_annules=("CANCELLED", "mean"),
)
par_aeroport["pct_annules"] = (par_aeroport["pct_annules"] * 100).round(2)
par_aeroport["pct_depart_retard"] = (flights.loc[a_un_depart, "DEPARTURE_DELAY"].ge(15).groupby(flights.loc[a_un_depart, "ORIGIN_AIRPORT"]).mean() * 100).round(1)
par_aeroport["pct_arrivee_retard"] = (flights[arrive].groupby("DESTINATION_AIRPORT")["EN_RETARD"].mean() * 100).round(1)
par_aeroport["ville"] = airports.set_index("IATA_CODE")["CITY"]

par_aeroport.nlargest(30, "vols").sort_values("pct_depart_retard", ascending=False)

**Constat :** parmi les 30 plus gros aéroports, la part de départs en retard va de 12,4 % (Salt Lake City) à 24,1 % (Baltimore). New York LaGuardia est le pire à l'arrivée (25,5 %) et annule le plus (4,30 %). Atlanta, le plus gros aéroport, reste à 17,3 %.

**Limite :** un aéroport accueille surtout certaines compagnies, et les écarts entre aéroports peuvent donc aussi venir d'elles.

### Les plus gros aéroports sont-ils les moins ponctuels ?

On mesure le lien entre la taille et le retard avec une corrélation de rang, sur les aéroports d'au moins 10 000 vols au départ. Elle va de -1 à 1 : 0 veut dire aucun lien, et 1 veut dire que le retard augmente toujours avec la taille.

In [ ]:
aeroports_comparables = par_aeroport[par_aeroport["vols"] >= 10_000]

len(aeroports_comparables), aeroports_comparables["vols"].rank().corr(aeroports_comparables["pct_depart_retard"].rank()).round(2)

**Constat :** sur 80 aéroports, la corrélation est de 0,56 : les gros aéroports ont tendance à avoir plus de départs en retard, mais la taille n'explique pas tout, comme le montre Atlanta.

## 3. Storyboard

Le plan de l'histoire racontée dans l'app : une page, une question, un message.

| Page | Question | Message clé | Visuel envisagé |
|---|---|---|---|
| 1. L'essentiel | Quelle est l'ampleur ? | Près d'un vol sur cinq n'arrive pas à l'heure | 4 chiffres clés et une barre « 100 vols » |
| 2. Quand ? | À quel moment le risque monte-t-il ? | Le risque grimpe de 9 % à 6 h à 26 % à 20 h, et février et juin sont les mois les plus difficiles | Courbe par heure et barres par mois |
| 3. Pourquoi ? | D'où viennent les retards ? | La première cause est l'effet domino : un avion en retard repart en retard | Barres des causes et courbe de l'effet domino par heure |
| 4. Avec qui et où ? | Qu'est-ce qui change le risque ? | La compagnie et l'aéroport comptent, mais la taille n'explique pas tout | Classement des compagnies et carte des aéroports |

**Indicateur principal :** la part de vols en retard de 15 minutes ou plus.

**À nuancer à l'écran :** les écarts entre compagnies et aéroports ne sont pas une note de qualité, la météo de juin est une explication probable mais pas prouvée, la colonne « météo extrême » sous-estime la météo, et tous les chiffres portent uniquement sur 2015.

### Le risque de retard monte-t-il vraiment au fil de la journée ?

La page 2 a besoin de la part de vols en retard par heure de départ prévue. Jusqu'ici, on n'avait que des minutes par heure. On garde les heures d'au moins 10 000 vols arrivés : de 1 h à 4 h, il y a trop peu de vols pour une comparaison fiable.

In [ ]:
par_heure = flights[arrive].groupby("HEURE_DEPART_PREVUE")["EN_RETARD"].agg(vols="size", pct_en_retard="mean")
par_heure["pct_en_retard"] = (par_heure["pct_en_retard"] * 100).round(1)

par_heure[par_heure["vols"] >= 10_000]

**Constat :** la part de vols en retard monte presque sans interruption, de 7,3 % à 5 h et 9,0 % à 6 h jusqu'à 26,3 % à 20 h, puis redescend à 19,2 % à 23 h.

### Le risque du soir touche-t-il toutes les compagnies ?

Le filtre de la page 2 n'a d'intérêt que si la réponse est lisible pour chaque compagnie. On compare donc 6 h et 20 h.

In [ ]:
matin_soir = flights[arrive & flights["HEURE_DEPART_PREVUE"].isin([6, 20])]
retard_matin_soir = (matin_soir.groupby(["AIRLINE", "HEURE_DEPART_PREVUE"])["EN_RETARD"].mean() * 100).unstack().round(1)
retard_matin_soir["vols_20h"] = matin_soir[matin_soir["HEURE_DEPART_PREVUE"] == 20].groupby("AIRLINE").size()

retard_matin_soir.rename(index=noms_compagnies).sort_values(20)

**Constat :** le risque est plus élevé à 20 h qu'à 6 h pour les 14 compagnies, mais pas avec la même ampleur : +8,2 points pour American Eagle, +25,9 points pour Southwest. Le filtre par compagnie de la page 2 a donc un intérêt.

**Limite :** ces écarts ne mesurent pas que la compagnie, car leurs réseaux et leurs horaires sont différents.

### Tous les pourcentages ont-ils la même base ?

Le 18,3 % de la vue d'ensemble compte tous les vols, alors que les tableaux par mois, compagnie et heure ne comptent que les vols arrivés. On mesure l'écart entre les deux.

In [ ]:
pd.Series({
    "sur tous les vols": flights["EN_RETARD"].mean(),
    "sur les vols arrivés": flights.loc[arrive, "EN_RETARD"].mean(),
}).mul(100).round(2)

**Constat :** l'écart est de 0,33 point (18,28 % contre 18,61 %). Il vient des vols annulés et déviés, qui n'ont pas de retard à l'arrivée.

**Choix :** la page 1 garde 18,3 % sur tous les vols, pour que ses 4 catégories fassent 100 %. Les détails par heure, mois et compagnie portent sur les vols arrivés, et l'app l'indiquera sous les graphiques.